# Case study: the January 2012 Pacific Northwest storm

## Purpose

How do the initialization approaches represent the 16-20 January 2012 Pacific Northwest storm in the Dec 2011 starts, under prescribed observed SST? The storm brought heavy snow inland and in the Cascades, record rain and flooding in western Oregon (an atmospheric-river, "Pineapple Express" episode) and hurricane-force gusts on the coast. This notebook does the first steps: the simulation coverage, regional daily temperature and precipitation of every member against the observations on a fixed observed window, member event metrics, event maps, and the large-scale storm environment (sea-level pressure, 500-hPa height, integrated vapour transport) before and during the event.

At leads of 2-5 weeks the question is whether an experiment produces **a favourable regional circulation and cold-wet environment during the observed period**, not whether it reproduces the exact cyclone track; the environment maps keep the two apart.

## Design: three questions (as notebook 06)

| Question | Experiments |
|---|---|
| Q1. Does event-informed initialization improve the event? | HYB_BASE and DA_COUPLED against CLIM_BASE (free-running-reference baseline) |
| Q2. Do the two initialization approaches differ? | DA_COUPLED against HYB_BASE |
| Q3. What roles do the component states and their consistency play? | DA_ATM (DART atmosphere, reference land) and DA_LND (ERA5 atmosphere, DART land) |

`PRIMARY` holds the Q1/Q2 experiments. All are 20-member ensembles; the series and maps pool the 4 starts (80 members); the metrics keep each start (one marker per start).

**Caveat (2026-11-06):** HYB_BASE and DA_LND started from an ERA5 atmosphere without the radiative tracers, aerosols and chemistry (6-7 K too cold over land for weeks); they are dashed and marked "IC defect" (`IC_DEFECT`) until rerun from the completed initial files. Q2 and the DA_LND part of Q3 wait for those reruns.

## Event definition

Fixed observed window `EVENT_WINDOW` = 16-20 Jan 2012 (antecedent period from the start to 15 Jan; aftermath 21-23 Jan). Regions (land, 1-degree): `WEST_WA_OR` (42-49N, 124.5-121.5W: coast, Puget-Willamette lowlands and the western Cascades) and `INLAND_PNW` (42-49N, 121.5-111W: east of the Cascades to Idaho). Observations: ERA5 daily-mean 2-m temperature and GPCP V3.3 daily precipitation (1 degree); no gridded snow observation is used (the archive's SWE products are monthly and mask mountains), so model snow water (H2OSNO) is shown without observations.

## Metrics (every member, start and region)

On the event window: event-total and maximum daily precipitation, peak-precipitation date error (searched within +-`PEAK_TOLERANCE_DAYS`), event-mean and minimum daily-mean temperature, and **cold-wet coincidence**: precipitation on days with daily-mean temperature at or below 0 C and +1 C (not diagnosed snowfall or freezing rain). Spatial: pattern correlation and RMSE over the two regions of event-total precipitation and event-mean temperature. Temperature is used raw (`none`) and with the model's mean bias removed (`amip`: minus the AMIP-minus-observed climatology of the day), so the freezing counts are not set by the model's climate offset (`BIASES`).

## Outputs

`data/hind_lead_eval/winter_storm_2012/`: `box_<CONFIG>_DJF_<VAR>.nc` (member fields on the boxes), `coverage.csv`, `storm_metrics.csv` (members), `storm_metrics_obs.csv`; figures in `figure/hind_lead_eval/winter_storm_2012/`.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths, experiments, event, regions and options. Plain values only, so `papermill -p` overrides work.

In [2]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                        # free-running E3SMv3 DECK AMIP runs
WORKFLOW_NAME = "hind_lead_eval"

CONFIGS = ["CLIM_BASE", "HYB_BASE", "DA_COUPLED", "DA_ATM", "DA_LND"]
PRIMARY = ["CLIM_BASE", "HYB_BASE", "DA_COUPLED"]
IC_DEFECT = ["HYB_BASE", "DA_LND"]   # started without radiative tracers; [] once rerun
SEASON = "DJF"                       # the Dec 2011 starts
N_DAYS = 61                          # simulated days used (lead 1 = start date)
MAX_MEMBERS = None
REQUIRE_ALL_MEMBERS = True

EVENT_WINDOW = ("2012-01-16", "2012-01-20")
SERIES_XLIM = ("2012-01-06", "2012-01-27")
OBS_PERIOD = ("2011-12-16", "2012-02-29")    # covers every start's simulated days
MAP_BOX = dict(lat=(39.5, 52.5), lon=(227.5, 250.5))   # Pacific Northwest
ENV_BOX = dict(lat=(20.5, 65.5), lon=(180.5, 255.5))   # North Pacific to the Rockies
MAP_EXTENT = (-130.0, -110.0, 40.0, 52.0)
ENV_EXTENT = (-178.0, -106.0, 22.0, 64.0)
REGIONS = [dict(name="WEST_WA_OR", lat=(42.0, 49.0), lon=(235.5, 238.5), mask="land"),
           dict(name="INLAND_PNW", lat=(42.0, 49.0), lon=(238.5, 249.0), mask="land")]
PEAK_REGION = "WEST_WA_OR"           # region whose observed peak sets the timing reference
ENV_DAYS = ["2012-01-14", "2012-01-16", "2012-01-18", "2012-01-20"]
CLIM_YEARS = (2001, 2014)
AMIP_CASES = ("v3.LR.amip_0101", "v3.LR.amip_0151")
PEAK_TOLERANCE_DAYS = 3
BIASES = ("none", "amip")
SPREAD = (0.1, 0.9)

FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 200

## 3. Paths and modules

In [3]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import STYLE_MAP, balanced_runs, check_configs, init_dates, readiness
from configs.fcst_init_experiment_config import members as config_members
from configs.output_paths import workflow_output_dirs

check_configs(CONFIGS)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, "winter_storm_2012", output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(data_root=INPUT_DATA_ROOT, land_support_dir=LAND_SUPPORT_DIR)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")


def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

[DATA] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/winter_storm_2012
[FIGURE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/figure/hind_lead_eval/winter_storm_2012


## 4. Read input

In [4]:
status = readiness(CONFIGS, [SEASON], streams=(("atm", "daily"), ("atm", "6hourly"), ("lnd", "daily")),
                   data_root=INPUT_DATA_ROOT)
RUNS = balanced_runs(status, REQUIRE_ALL_MEMBERS)
inits = list(init_dates([SEASON])[SEASON])
members = {c: (lambda m: m if MAX_MEMBERS is None else m[:MAX_MEMBERS])(config_members(c, SEASON)) for c in CONFIGS}
print(f"[RUNS] {RUNS}\n[STARTS] {[str(i) for i in inits]}\n[MEMBERS] {({c: len(m) for c, m in members.items()})}")

[RUNS] [('CLIM_BASE', 'DJF'), ('DA_ATM', 'DJF'), ('DA_COUPLED', 'DJF'), ('DA_LND', 'DJF'), ('HYB_BASE', 'DJF')]
[STARTS] ['2011-12-16', '2011-12-21', '2011-12-26', '2011-12-31']
[MEMBERS] {'CLIM_BASE': 20, 'HYB_BASE': 20, 'DA_COUPLED': 20, 'DA_ATM': 20, 'DA_LND': 20}


## 5. Observations, climatologies and coverage

ERA5 daily-mean TREFHT and GPCP precipitation on the map box; both TREFHT climatologies (for the `amip` bias treatment and the temperature anomaly maps); regional masks; the observed event metrics and the simulation coverage of the window.

In [ ]:
from util.fcst_init_data import region_masks
from util.heatwave_case import climatologies, coverage_table, observed_box, region_mean, region_weights
from util.winter_storm_case import KELVIN, pattern_scores, storm_metrics

years = range(CLIM_YEARS[0], CLIM_YEARS[1] + 1)
obs_T = observed_box("TREFHT", *OBS_PERIOD, MAP_BOX, observation_root=OBSERVATION_ROOT) - KELVIN
obs_P = observed_box("PRECT", *OBS_PERIOD, MAP_BOX, observation_root=OBSERVATION_ROOT)
clim = climatologies("TREFHT", *OBS_PERIOD, years, MAP_BOX, observation_root=OBSERVATION_ROOT, amip_root=AMIP_ROOT,
                     amip_cases=AMIP_CASES, land_support_dir=LAND_SUPPORT_DIR)
bias_map = clim.clim_amip - clim.clim_obs   # model mean bias of each day (K)
masks = region_masks(REGIONS, obs_T.lat.values, obs_T.lon.values, land_support_dir=LAND_SUPPORT_DIR)
W = {r["name"]: region_weights(masks.sel(region=r["name"])) for r in REGIONS}
W_ALL = region_weights(masks.any("region"))
print({k: int((v > 0).sum()) for k, v in W.items()}, "land points")

obs_reg = {r: dict(TREFHT=region_mean(obs_T, w), PRECT=region_mean(obs_P, w), H2OSNO=None) for r, w in W.items()}
times = obs_T.time.values
pk = obs_reg[PEAK_REGION]["PRECT"].sel(time=slice(*EVENT_WINDOW))
obs_peak = str(pd.Timestamp(pk.time.values[int(np.argmax(pk.values))]).date())
obs_event_T = obs_T.sel(time=slice(*EVENT_WINDOW)).mean("time")
obs_event_P = obs_P.sel(time=slice(*EVENT_WINDOW)).sum("time")
obs_rows = []
for r in W:
    m = storm_metrics(obs_reg[r]["TREFHT"].values, obs_reg[r]["PRECT"].values, times, EVENT_WINDOW, obs_peak,
                      tolerance_days=PEAK_TOLERANCE_DAYS)
    obs_rows.append(dict(region=r, **m))
obs_metrics = pd.DataFrame(obs_rows).set_index("region")
obs_metrics.to_csv(data_dir / "storm_metrics_obs.csv")
cov = coverage_table(inits, N_DAYS, EVENT_WINDOW)
cov.to_csv(data_dir / "coverage.csv", index=False)
print(f"[EVENT] {EVENT_WINDOW[0]} to {EVENT_WINDOW[1]}; observed {PEAK_REGION} precipitation peak {obs_peak}")
display(obs_metrics.round(2), cov)

{'WEST_WA_OR': 24, 'INLAND_PNW': 77} land points
[EVENT] 2012-01-16 to 2012-01-20; observed WEST_WA_OR precipitation peak 2012-01-19


,event_precip_mm,max_daily_precip_mm,peak_precip_date_error_days,event_mean_t_c,event_min_t_c,precip_freezing_days_mm,precip_near_freezing_days_mm,window_days_covered
region,,,,,,,,
WEST_WA_OR,97.89,48.14,0,0.26,-3.34,3.12,31.10,5
INLAND_PNW,21.72,10.43,2,-6.36,-9.64,21.72,21.72,5


,init,last_day,lead_at_onset,lead_at_end,window_covered
0,2011-12-16,2012-02-14,32,36,full
1,2011-12-21,2012-02-19,27,31,full
2,2011-12-26,2012-02-24,22,26,full
3,2011-12-31,2012-02-29,17,21,full


: 

## 6. Model fields

Daily TREFHT, PRECT and H2OSNO of every member on the map box, and PSL, Z500, TUQ, TVQ on the environment box (cached; reused unless `FORCE_COMPUTE`), put on calendar days. Z500 is the 00Z state ending each day, in the model and in ERA5.

In [ ]:
from util.heatwave_case import calendar, model_box

MAP_VARS, ENV_VARS = ("TREFHT", "PRECT", "H2OSNO"), ("PSL", "Z500", "TUQ", "TVQ")
fields = {}
for c in CONFIGS:
    for v, box in [(v, MAP_BOX) for v in MAP_VARS] + [(v, ENV_BOX) for v in ENV_VARS]:
        tag = "map" if box is MAP_BOX else "env"
        fields[c, v] = model_box(c, SEASON, v, members[c], box, n_days=N_DAYS,
                                 cache=data_dir / f"box_{tag}_{c}_{SEASON}_{v}.nc", force=FORCE_COMPUTE, **ROOTS)
    print(f"[MODEL] {c}: " + ", ".join(f"{v} {dict(fields[c, v].sizes)['member']}x{dict(fields[c, v].sizes)['lead']}"
                                       for v in MAP_VARS + ENV_VARS))


def pooled(da):
    """(member, time, lat, lon) on calendar days within OBS_PERIOD, members of all starts stacked
    (member labels made unique per start); days a start does not cover are NaN."""
    days = pd.date_range(*OBS_PERIOD, freq="D").values
    out = []
    for init in inits:
        x = da.sel(init=np.datetime64(init, "ns")).drop_vars("init", errors="ignore")
        x = x.assign_coords(lead=calendar(init, x.lead.values).astype("datetime64[ns]")).rename(lead="time")
        x = x.reindex(time=days).assign_coords(member=[f"{str(init)[5:10]}_{m}" for m in x.member.values])
        out.append(x)
    return xr.concat(out, "member")

[MODEL] CLIM_BASE: TREFHT 20x61, PRECT 20x61, H2OSNO 20x61, PSL 20x61, Z500 20x61, TUQ 20x61, TVQ 20x61


## 7. Regional series and metrics

In [ ]:
T_raw = {c: pooled(fields[c, "TREFHT"]) - KELVIN for c in CONFIGS}
P_all = {c: pooled(fields[c, "PRECT"]) for c in CONFIGS}
S_all = {c: pooled(fields[c, "H2OSNO"]) for c in CONFIGS}
start_of = {c: np.array([m.split("_")[0] for m in T_raw[c].member.values]) for c in CONFIGS}

rows, series = [], {}
for b in BIASES:
    for c in CONFIGS:
        T = T_raw[c] - (bias_map if b == "amip" else 0.0)
        ev_T = T.sel(time=slice(*EVENT_WINDOW)).mean("time")
        ev_P = P_all[c].sel(time=slice(*EVENT_WINDOW)).sum("time")
        for r, w in W.items():
            series[b, r, c] = dict(TREFHT=region_mean(T, w), PRECT=region_mean(P_all[c], w), H2OSNO=region_mean(S_all[c], w))
        for k, m in enumerate(T.member.values):
            spat = dict(zip(("precip_pattern_r", "precip_pattern_rmse_mm"),
                            pattern_scores(ev_P.isel(member=k), obs_event_P, W_ALL)))
            spat.update(zip(("t_pattern_r", "t_pattern_rmse_c"), pattern_scores(ev_T.isel(member=k), obs_event_T, W_ALL)))
            for r in W:
                s = series[b, r, c]
                met = storm_metrics(s["TREFHT"].isel(member=k).values, s["PRECT"].isel(member=k).values, times,
                                    EVENT_WINDOW, obs_peak, tolerance_days=PEAK_TOLERANCE_DAYS)
                rows.append(dict(bias=b, config=c, init=start_of[c][k], member=str(m), region=r, **met, **spat))
metrics = pd.DataFrame(rows)
metrics.to_csv(data_dir / "storm_metrics.csv", index=False)
print(f"[WRITE] {data_dir / 'storm_metrics.csv'}")
obs_spat = dict(precip_pattern_r=1.0, precip_pattern_rmse_mm=0.0, t_pattern_r=1.0, t_pattern_rmse_c=0.0)
summary = metrics.groupby(["bias", "region", "config"]).median(numeric_only=True).reindex(CONFIGS, level="config")
display(summary.round(2))
display(obs_metrics.round(2))

## 8. Plot and show

Regional series (Q1/Q2 set, then all five), member metrics per region, event maps of precipitation and temperature anomaly, and the storm environment.

In [ ]:
from util.heatwave_case import plot_metrics
from util.winter_storm_case import plot_environment, plot_event_maps, plot_region_series

ylabels = {"TREFHT": "daily-mean T2m (C)", "PRECT": "precipitation (mm/day)", "H2OSNO": "snow water (mm, model)"}
for b in BIASES:
    for tag, cfgs in (("primary", PRIMARY), ("all", CONFIGS)):
        panels = {r: {v: {c: series[b, r, c][v] for c in cfgs} for v in ("TREFHT", "PRECT", "H2OSNO")} for r in W}
        save(plot_region_series(panels, obs_reg, EVENT_WINDOW, STYLE_MAP, IC_DEFECT, ylabels=ylabels, spread=SPREAD,
                                xlim=SERIES_XLIM,
                                title=f"Regional daily means, ensemble mean and {int(SPREAD[0] * 100)}-{int(SPREAD[1] * 100)}% "
                                      f"range of the 80 members of the 4 Dec starts (temperature: {b}); OBS = ERA5, GPCP"),
             f"storm_series_{tag}_{b}.png")

METRICS = ["event_precip_mm", "max_daily_precip_mm", "peak_precip_date_error_days", "event_mean_t_c", "event_min_t_c",
           "precip_freezing_days_mm", "precip_near_freezing_days_mm", "precip_pattern_r", "t_pattern_r"]
for b in BIASES:
    for r in W:
        tab = metrics[(metrics.bias == b) & (metrics.region == r)]
        obs_m = {**obs_metrics.loc[r].to_dict(), **obs_spat}
        save(plot_metrics(tab, obs_m, METRICS, CONFIGS, STYLE_MAP, IC_DEFECT,
                          title=f"Storm metrics {EVENT_WINDOW[0]} to {EVENT_WINDOW[1]}, {r} (members, one marker per start; "
                                f"bar = median; black = ERA5/GPCP; temperature: {b})"),
             f"storm_metrics_{r}_{b}.png")

model_P = {c: P_all[c].sel(time=slice(*EVENT_WINDOW)).sum("time").mean("member") for c in CONFIGS}
save(plot_event_maps(obs_event_P, model_P, REGIONS, IC_DEFECT, cmap="GnBu", vmin=0, vmax=120, units="event total (mm)",
                     extent=MAP_EXTENT, title=f"Precipitation {EVENT_WINDOW[0]} to {EVENT_WINDOW[1]} (GPCP; ensemble and start mean)"),
     "storm_map_precip.png")
obs_anom = (obs_T + KELVIN - clim.clim_obs).sel(time=slice(*EVENT_WINDOW)).mean("time")
for b in BIASES:
    ref = clim.clim_amip if b == "amip" else clim.clim_obs
    model_T = {c: (T_raw[c] + KELVIN - ref).sel(time=slice(*EVENT_WINDOW)).mean(("time", "member")) for c in CONFIGS}
    save(plot_event_maps(obs_anom, model_T, REGIONS, IC_DEFECT, cmap="RdBu_r", vmin=-8, vmax=8, units="anomaly (K)",
                         extent=MAP_EXTENT, title=f"T2m anomaly {EVENT_WINDOW[0]} to {EVENT_WINDOW[1]} (vs {b} climatology)"),
         f"storm_map_t2m_{b}.png")

### Storm environment

Ensemble and start mean of each configuration on `ENV_DAYS` against ERA5: IVT magnitude (shading) and vectors, Z500 (black, 60 m), PSL (white, 8 hPa). A long-lead ensemble mean smooths individual cyclones; it shows whether the experiment's mean flow favours onshore moisture transport into the Pacific Northwest during the observed period.

In [ ]:
env_days = pd.to_datetime(ENV_DAYS)
obs_env = {}
for v in ENV_VARS:
    if v == "Z500":   # 00Z state ending each day: read the next day's 00Z and label it with the day
        x = observed_box(v, str((env_days.min() + pd.Timedelta(days=1)).date()),
                         str((env_days.max() + pd.Timedelta(days=1)).date()), ENV_BOX, observation_root=OBSERVATION_ROOT)
        x = x.assign_coords(time=x.time.values - np.timedelta64(1, "D"))
    else:
        x = observed_box(v, str(env_days.min().date()), str(env_days.max().date()), ENV_BOX, observation_root=OBSERVATION_ROOT)
    obs_env[v] = x
obs_days = {d: xr.Dataset({v: obs_env[v].sel(time=d) for v in ENV_VARS}) for d in env_days}
model_days = {}
for c in CONFIGS:
    pool = {v: pooled(fields[c, v]).sel(time=env_days.values).mean("member") for v in ENV_VARS}
    model_days[c] = {d: xr.Dataset({v: pool[v].sel(time=d) for v in ENV_VARS}) for d in env_days}
for tag, cfgs in (("primary", PRIMARY), ("all", CONFIGS)):
    save(plot_environment(list(env_days), obs_days, {c: model_days[c] for c in cfgs}, IC_DEFECT, extent=ENV_EXTENT,
                          title="Storm environment: ERA5 and ensemble-and-start means (IVT shading/vectors, Z500 black, PSL white)"),
         f"storm_environment_{tag}.png")